# TACO: prompts por dificultad

Lee Arrow por lotes sin necesitar el parquet intermedio. Cada salida contiene solo `prompt`. Se conservan todas las dificultades, incluidas las desconocidas.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "modeling/src/data/extract_prompts.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "modeling/data"


In [2]:
import pyarrow as pa
from collections import defaultdict
from modeling.src.data.extract_prompts import export_prompts
levels = defaultdict(list)
for split in ("train", "test"):
    shards = sorted((DATA / "raw/TACO" / split).glob("*.arrow"))
    if not shards:
        raise FileNotFoundError(f"Faltan archivos TACO: {split}")
    for shard in shards:
        with pa.memory_map(str(shard), "r") as source:
            for batch in pa.ipc.open_stream(source):
                for row in batch.select(["question", "difficulty"]).to_pylist():
                    levels[str(row["difficulty"] or "UNKNOWN_DIFFICULTY").upper()].append(row["question"])
OUT = DATA / "processed/por_nivel"
for level, prompts in sorted(levels.items()):
    # El nombre procede de la etiqueta, sin permitir rutas fuera de OUT.
    filename = "".join(c if c.isalnum() or c == "_" else "_" for c in level.lower())
    print(level, export_prompts(prompts, OUT / f"{filename}.jsonl"))


EASY {'read': 9104, 'exported': 9074, 'empty_or_short': 0, 'duplicates': 30}
HARD {'read': 3362, 'exported': 3189, 'empty_or_short': 0, 'duplicates': 173}
MEDIUM {'read': 3444, 'exported': 3416, 'empty_or_short': 0, 'duplicates': 28}
MEDIUM_HARD {'read': 2945, 'exported': 2884, 'empty_or_short': 0, 'duplicates': 61}
UNKNOWN_DIFFICULTY {'read': 5014, 'exported': 4880, 'empty_or_short': 0, 'duplicates': 134}
VERY_HARD {'read': 2574, 'exported': 2395, 'empty_or_short': 0, 'duplicates': 179}
